# Hydra Ablation Experiments

## 1. Setup and Imports

## 2. Baseline Verification

## 3. Experiment 1 - Max-Only vs. Max-and-Min

## 4. Experiment 2 - Hard vs. Soft vs. Combined Counting

## 5. Experiment 3 - First-Order Differencing On vs. Off

## 6. Experiment 4 - Clipping On vs. Off

## 7. Experiment 5 - Sensitivity Analysis

## 8. Results Summary and Visualisations

## 9. Discussion and Conclusion


In [3]:

from pathlib import Path
import sys
import time
import importlib.util

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score, f1_score

# Project folders
PROJECT_DIR = Path.cwd().parent

REPO_DIR = PROJECT_DIR / "hydra-authors-repo"
HYDRA_SOURCE = REPO_DIR / "code" / "hydra.py"

DATA_DIR = PROJECT_DIR / "data" / "ucr_tsml"
RESULTS_DIR = PROJECT_DIR / "results" / "Mahak"

# Create the results folder if it doesn't exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Check paths before continuing
print("Project directory:", PROJECT_DIR)
print("Hydra source exists:", HYDRA_SOURCE.exists())
print("Results directory:", RESULTS_DIR)
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

if not HYDRA_SOURCE.exists():
    raise FileNotFoundError(
        f"Could not find {HYDRA_SOURCE}. "
        "Check that the notebook is running from your project root."
    )

# Load the original Hydra implementation without modifying its file
spec = importlib.util.spec_from_file_location(
    "hydra_official_impl", HYDRA_SOURCE
)
hydra_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = hydra_module
spec.loader.exec_module(hydra_module)

Hydra = hydra_module.Hydra
SparseScaler = hydra_module.SparseScaler

print("\nSetup successful!")
print("Original Hydra implementation loaded.")



Project directory: /Users/shreed/PROJECTS/Hydra_1
Hydra source exists: True
Results directory: /Users/shreed/PROJECTS/Hydra_1/results/Mahak
PyTorch version: 2.15.0
NumPy version: 2.3.5
Pandas version: 3.0.6

Setup successful!
Original Hydra implementation loaded.


In [4]:

# Experiment 1: Max-only vs. Max-and-min

# Start with these three manageable datasets.
# We can expand the experiment after the first run works.
ABLATION_DATASETS = ["Coffee", "ECG200", "GunPoint"]

RANDOM_SEEDS = [0]
BATCH_SIZE = 256

print("Experiment 1 datasets:", ABLATION_DATASETS)
print("Random seeds:", RANDOM_SEEDS)
print("Batch size:", BATCH_SIZE)

print("\nConfiguration ready.")



Experiment 1 datasets: ['Coffee', 'ECG200', 'GunPoint']
Random seeds: [0]
Batch size: 256

Configuration ready.


In [5]:
from aeon.datasets import load_classification
import numpy as np


def load_ucr_dataset(dataset_name, extract_path=DATA_DIR):
    """
    Load the predefined train/test split for a UCR classification dataset.

    Returns
    -------
    X_train : np.ndarray
        Shape: (n_train, n_channels, n_timepoints)
    y_train : np.ndarray
        Training labels.
    X_test : np.ndarray
        Shape: (n_test, n_channels, n_timepoints)
    y_test : np.ndarray
        Test labels.
    """
    X_train, y_train = load_classification(
        name=dataset_name,
        split="train",
        extract_path=str(extract_path),
        return_metadata=False,
    )

    X_test, y_test = load_classification(
        name=dataset_name,
        split="test",
        extract_path=str(extract_path),
        return_metadata=False,
    )

    X_train = np.asarray(X_train, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)

    # Consistent label representation across train and test.
    y_train = np.asarray(y_train).astype(str)
    y_test = np.asarray(y_test).astype(str)

    if X_train.ndim != 3 or X_test.ndim != 3:
        raise ValueError(
            f"{dataset_name}: Expected 3D input arrays. "
            f"Got train={X_train.shape}, test={X_test.shape}"
        )

    if X_train.shape[1] != 1 or X_test.shape[1] != 1:
        raise ValueError(
            f"{dataset_name}: Expected univariate data for this "
            f"baseline. Train={X_train.shape}, test={X_test.shape}"
        )

    if X_train.shape[2] != X_test.shape[2]:
        raise ValueError(
            f"{dataset_name}: Train/test series lengths differ."
        )

    if len(X_train) != len(y_train) or len(X_test) != len(y_test):
        raise ValueError(
            f"{dataset_name}: Number of samples does not match labels."
        )

    if not set(np.unique(y_test)).issubset(set(np.unique(y_train))):
        raise ValueError(
            f"{dataset_name}: Test labels include classes absent from training."
        )

    return X_train, y_train, X_test, y_test

In [6]:

# Test that the dataset loader works correctly
X_train, y_train, X_test, y_test = load_ucr_dataset("Coffee")

print("Dataset: Coffee")
print("Training data shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Test data shape:", X_test.shape)
print("Test labels shape:", y_test.shape)
print("Training classes:", np.unique(y_train))
print("Test classes:", np.unique(y_test))



Dataset: Coffee
Training data shape: (28, 1, 286)
Training labels shape: (28,)
Test data shape: (28, 1, 286)
Test labels shape: (28,)
Training classes: ['0' '1']
Test classes: ['0' '1']


In [7]:

# Experiment 1: Inspect the original Hydra configuration

import inspect

print("Hydra constructor:")
print(inspect.signature(Hydra))

print("\nOriginal Hydra forward method:")
print(inspect.getsource(Hydra.forward))



Hydra constructor:
(input_length, k=8, g=64, seed=None)

Original Hydra forward method:
    def forward(self, X):

        num_examples = X.shape[0]

        if self.divisor > 1:
            diff_X = torch.diff(X)

        Z = []

        for dilation_index in range(self.num_dilations):

            d = self.dilations[dilation_index].item()
            p = self.paddings[dilation_index].item()

            for diff_index in range(self.divisor):

                _Z = F.conv1d(X if diff_index == 0 else diff_X, self.W[dilation_index, diff_index], dilation = d, padding = p) \
                      .view(num_examples, self.h, self.k, -1)

                max_values, max_indices = _Z.max(2)
                count_max = torch.zeros(num_examples, self.h, self.k)

                min_values, min_indices = _Z.min(2)
                count_min = torch.zeros(num_examples, self.h, self.k)

                count_max.scatter_add_(-1, max_indices, max_values)
                count_min.scatter_add_(-1, mi

In [8]:

class HydraMaxOnly(Hydra):

    def forward(self, X):
        num_examples = X.shape[0]

        if self.divisor > 1:
            diff_X = torch.diff(X)

        Z = []

        for dilation_index in range(self.num_dilations):
            d = self.dilations[dilation_index].item()
            p = self.paddings[dilation_index].item()

            for diff_index in range(self.divisor):
                input_X = X if diff_index == 0 else diff_X

                conv = torch.nn.functional.conv1d(
                    input_X,
                    self.W[dilation_index, diff_index],
                    dilation=d,
                    padding=p,
                )

                conv = conv.view(
                    num_examples, self.h, self.k, -1
                )

                max_values, max_indices = conv.max(2)

                count_max = torch.zeros(
                    num_examples,
                    self.h,
                    self.k,
                    device=conv.device,
                    dtype=conv.dtype,
                )

                count_max.scatter_add_(
                    -1, max_indices, max_values
                )

                Z.append(count_max)

        return torch.cat(Z, 1).reshape(num_examples, -1)


print("Max-only Hydra class created successfully.")



Max-only Hydra class created successfully.


In [9]:

# Load the Coffee dataset
X_train, y_train, X_test, y_test = load_ucr_dataset("Coffee")

# Use four examples for a quick test
X_sample = torch.tensor(X_train[:4], dtype=torch.float32)

# Create both models with the same settings and seed
hydra_original = Hydra(
    input_length=X_train.shape[-1],
    k=8,
    g=64,
    seed=0,
)

hydra_max_only = HydraMaxOnly(
    input_length=X_train.shape[-1],
    k=8,
    g=64,
    seed=0,
)

# Generate features without calculating gradients
with torch.no_grad():
    Z_original = hydra_original(X_sample)
    Z_max_only = hydra_max_only(X_sample)

print("Original feature shape:", tuple(Z_original.shape))
print("Max-only feature shape:", tuple(Z_max_only.shape))

assert Z_original.shape[0] == Z_max_only.shape[0]
assert Z_original.shape[1] == 2 * Z_max_only.shape[1]

print("\nSUCCESS: Both Hydra versions produced the expected feature shapes.")




Original feature shape: (4, 6144)
Max-only feature shape: (4, 3072)

SUCCESS: Both Hydra versions produced the expected feature shapes.


In [10]:

# Experiment 1: Original Hydra vs. Max-Only Hydra

def evaluate_hydra_variant(
    dataset_name,
    variant_name,
    hydra_class,
    seed=0,
    k=8,
    g=64,
    batch_size=256,
):
    # Load the same predefined train/test split
    X_train, y_train, X_test, y_test = load_ucr_dataset(dataset_name)

    start_time = time.perf_counter()

    # Convert the data to PyTorch tensors
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

    # Create the selected Hydra transform
    model = hydra_class(
        input_length=X_train.shape[-1],
        k=k,
        g=g,
        seed=seed,
    )

    # Transform the data in batches
    with torch.no_grad():
        Z_train = model.batch(X_train_tensor, batch_size=batch_size)
        Z_test = model.batch(X_test_tensor, batch_size=batch_size)

    # Fit a separate scaler for this variant
    scaler = SparseScaler()
    Z_train = scaler.fit_transform(Z_train)
    Z_test = scaler.transform(Z_test)

    # Train and evaluate the classifier
    classifier = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    classifier.fit(Z_train.numpy(), y_train)

    predictions = classifier.predict(Z_test.numpy())

    elapsed = time.perf_counter() - start_time

    return {
        "dataset": dataset_name,
        "variant": variant_name,
        "seed": seed,
        "k": k,
        "g": g,
        "accuracy": accuracy_score(y_test, predictions),
        "macro_f1": f1_score(
            y_test, predictions, average="macro", zero_division=0
        ),
        "runtime_seconds": elapsed,
    }


# Run the two comparisons on Coffee
results_exp1 = []

for variant_name, hydra_class in [
    ("original_max_and_min", Hydra),
    ("max_only", HydraMaxOnly),
]:
    result = evaluate_hydra_variant(
        dataset_name="Coffee",
        variant_name=variant_name,
        hydra_class=hydra_class,
        seed=0,
    )
    results_exp1.append(result)

    print(
        f"{variant_name}: "
        f"accuracy={result['accuracy']:.4f}, "
        f"macro_f1={result['macro_f1']:.4f}, "
        f"runtime={result['runtime_seconds']:.2f}s"
    )

# Save results (overwrites any previous run)
output_path = RESULTS_DIR / "ablation_max_min.csv"
results_df = pd.DataFrame(results_exp1)

results_df.to_csv(output_path, index=False)

print("\nResults saved to:", output_path)
display(results_df)



original_max_and_min: accuracy=1.0000, macro_f1=1.0000, runtime=0.28s
max_only: accuracy=1.0000, macro_f1=1.0000, runtime=0.23s

Results saved to: /Users/shreed/PROJECTS/Hydra_1/results/Mahak/ablation_max_min.csv


,dataset,variant,seed,k,g,accuracy,macro_f1,runtime_seconds
0,Coffee,original_max_and_min,0,8,64,1.0,1.0,0.281694
1,Coffee,max_only,0,8,64,1.0,1.0,0.225340


### Experiment 1 - Max-Only vs. Max-and-Min

**Setup:** Coffee, seed 0, k = 8, g = 64, `RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))`.

| Variant | Accuracy | Macro F1 | Features | Runtime (s) |
| ------- | -------: | -------: | -------: | ----------: |
| Original Hydra (max + min) | 1.000 | 1.000 | 6144 | 0.282 |
| Max-only Hydra | 1.000 | 1.000 | 3072 | 0.225 |

**Observation:** Both variants classified the Coffee test set perfectly. The max-only variant uses half as many features and was about 20% faster in this run (0.225 s vs 0.282 s).

**Conclusion:** On Coffee, removing the minimum-response features did not reduce accuracy. Coffee is already at 100% accuracy, though, so this experiment cannot show whether the min features help. The variant needs testing on harder datasets and with more seeds before drawing a general conclusion.

**Results file:** `results/Mahak/ablation_max_min.csv`

## Experiment 2 - Hard vs. Soft vs. Combined Counting

**Objective:** Investigate how the counting method used to build Hydra features affects classification performance.

At each time point, within each group, the kernel with the largest (max) response and the kernel with the smallest (min) response each "win". The two channels can record a win in two ways:

* **Hard counting:** add 1 for each win (counts how often a kernel wins).
* **Soft counting:** add the kernel's activation value for each win.

Variants compared:

| Variant | Max channel | Min channel | Note |
| ------- | ----------- | ----------- | ---- |
| `hard_only` | hard | hard | |
| `soft_only` | soft | soft | |
| `combined_default` | soft | hard | **Same as the original Hydra implementation** |

**Evaluation:** accuracy, macro F1 and runtime, using the same datasets, train/test splits and classifier settings for every variant.

In [11]:
class HydraCountingVariant(Hydra):
    """Hydra with configurable hard/soft maximum and minimum counting."""

    def __init__(
        self,
        input_length,
        k=8,
        g=64,
        seed=None,
        max_counting="soft",
        min_counting="hard",
    ):
        super().__init__(input_length=input_length, k=k, g=g, seed=seed)

        if max_counting not in ("hard", "soft"):
            raise ValueError("max_counting must be 'hard' or 'soft'")

        if min_counting not in ("hard", "soft"):
            raise ValueError("min_counting must be 'hard' or 'soft'")

        self.max_counting = max_counting
        self.min_counting = min_counting

    def forward(self, X):
        num_examples = X.shape[0]

        if self.divisor > 1:
            diff_X = torch.diff(X)

        Z = []

        for dilation_index in range(self.num_dilations):
            d = self.dilations[dilation_index].item()
            p = self.paddings[dilation_index].item()

            for diff_index in range(self.divisor):
                input_X = X if diff_index == 0 else diff_X

                conv = torch.nn.functional.conv1d(
                    input_X,
                    self.W[dilation_index, diff_index],
                    dilation=d,
                    padding=p,
                )

                conv = conv.view(num_examples, self.h, self.k, -1)

                # Maximum response
                max_values, max_indices = conv.max(2)
                max_counts = torch.zeros(
                    num_examples, self.h, self.k,
                    device=conv.device,
                    dtype=conv.dtype,
                )

                if self.max_counting == "soft":
                    max_updates = max_values
                else:
                    max_updates = torch.ones_like(max_values)

                max_counts.scatter_add_(-1, max_indices, max_updates)
                Z.append(max_counts)

                # Minimum response
                min_values, min_indices = conv.min(2)
                min_counts = torch.zeros(
                    num_examples, self.h, self.k,
                    device=conv.device,
                    dtype=conv.dtype,
                )

                if self.min_counting == "soft":
                    min_updates = min_values
                else:
                    min_updates = torch.ones_like(min_values)

                min_counts.scatter_add_(-1, min_indices, min_updates)
                Z.append(min_counts)

        return torch.cat(Z, dim=1).reshape(num_examples, -1)



In [12]:

# Test the three counting configurations
counting_variants = {
    "hard_only": HydraCountingVariant(
        input_length=100,
        seed=0,
        max_counting="hard",
        min_counting="hard",
    ),
    "soft_only": HydraCountingVariant(
        input_length=100,
        seed=0,
        max_counting="soft",
        min_counting="soft",
    ),
    "combined_default": HydraCountingVariant(
        input_length=100,
        seed=0,
        max_counting="soft",
        min_counting="hard",
    ),
}

dummy_X = torch.randn(4, 1, 100)

for name, model in counting_variants.items():
    with torch.no_grad():
        features = model.batch(dummy_X)

    print(f"{name}: feature shape = {tuple(features.shape)}")

print("\nAll three counting configurations ran successfully.")


print(f"{name}: feature shape = {tuple(features.shape)}")


print("\nAll three counting configurations ran successfully.")


hard_only: feature shape = (4, 4096)
soft_only: feature shape = (4, 4096)
combined_default: feature shape = (4, 4096)

All three counting configurations ran successfully.
combined_default: feature shape = (4, 4096)

All three counting configurations ran successfully.


In [13]:

# Experiment 2: Hard vs. Soft vs. Combined Counting

COUNTING_RESULTS_FILE = RESULTS_DIR / "ablation_counting_methods_v2.csv"

counting_configs = {
    "hard_only": {
        "max_counting": "hard",
        "min_counting": "hard",
    },
    "soft_only": {
        "max_counting": "soft",
        "min_counting": "soft",
    },
    "combined_default": {
        "max_counting": "soft",
        "min_counting": "hard",
    },
}

counting_results = []

for dataset_name in ABLATION_DATASETS:
    for seed in RANDOM_SEEDS:
        for variant_name, config in counting_configs.items():
            print(f"Running {dataset_name} | {variant_name} | seed={seed}")

            def make_model(**kwargs):
                return HydraCountingVariant(
                    **kwargs,
                    max_counting=config["max_counting"],
                    min_counting=config["min_counting"],
                )

            result = evaluate_hydra_variant(
                dataset_name=dataset_name,
                variant_name=variant_name,
                hydra_class=make_model,
                seed=seed,
                k=8,
                g=64,
                batch_size=BATCH_SIZE,
            )

            counting_results.append(result)

print(f"\nTotal results collected: {len(counting_results)}")

counting_df = pd.DataFrame(counting_results)

counting_df.to_csv(COUNTING_RESULTS_FILE, index=False)

print(f"\nResults saved to: {COUNTING_RESULTS_FILE}")
display(counting_df)



Running Coffee | hard_only | seed=0
Running Coffee | soft_only | seed=0
Running Coffee | combined_default | seed=0
Running ECG200 | hard_only | seed=0
Running ECG200 | soft_only | seed=0
Running ECG200 | combined_default | seed=0
Running GunPoint | hard_only | seed=0
Running GunPoint | soft_only | seed=0
Running GunPoint | combined_default | seed=0

Total results collected: 9

Results saved to: /Users/shreed/PROJECTS/Hydra_1/results/Mahak/ablation_counting_methods_v2.csv


,dataset,variant,seed,k,g,accuracy,macro_f1,runtime_seconds
0,Coffee,hard_only,0,8,64,1.00,1.000000,0.272382
1,Coffee,soft_only,0,8,64,1.00,1.000000,0.258845
2,Coffee,combined_default,0,8,64,1.00,1.000000,0.258898
3,ECG200,hard_only,0,8,64,0.84,0.816176,0.215289
4,ECG200,soft_only,0,8,64,0.87,0.856179,0.210602
5,ECG200,combined_default,0,8,64,0.87,0.856179,0.205894
6,GunPoint,hard_only,0,8,64,1.00,1.000000,0.351097
7,GunPoint,soft_only,0,8,64,1.00,1.000000,0.340599
8,GunPoint,combined_default,0,8,64,1.00,1.000000,0.354609


### Results

Three counting variants were evaluated on Coffee, ECG200 and GunPoint using seed 0, with k = 8, g = 64 and `RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))`.

| Dataset | hard_only | soft_only | combined_default (original) |
| ------- | --------: | --------: | --------------------------: |
| Coffee | 1.00 (F1 1.000) | 1.00 (F1 1.000) | 1.00 (F1 1.000) |
| ECG200 | 0.84 (F1 0.816) | 0.87 (F1 0.856) | 0.87 (F1 0.856) |
| GunPoint | 1.00 (F1 1.000) | 1.00 (F1 1.000) | 1.00 (F1 1.000) |

Runtimes were similar for all variants (about 0.21–0.35 s per dataset).

### Conclusion

All three variants scored 100% on Coffee and GunPoint. On ECG200, `soft_only` and the original Hydra (`combined_default`) tied at 87% accuracy, with identical macro F1 (0.856). `hard_only` was 3 percentage points lower, at 84% (F1 0.816).

* **Hard counting on the max channel hurts accuracy.** The original choice of soft counting for the max channel is supported.
* **Switching the min channel from hard to soft counting made no difference** on these datasets.

Note: an earlier run used the classifier's default alphas and had a bug in the variant class. It reported `soft_only` at 90% on ECG200, beating the original. With the bug fixed and the standard alphas, that advantage disappears, so it should not be treated as an improvement.

The original Hydra here gives 0.87 on ECG200, which matches the baseline notebook (`shreed_hydra_baseline.ipynb`, seed 0). This confirms the pipelines are consistent.

These results use one seed and three datasets, two of which are already at 100% accuracy. More runs on harder datasets are needed for general conclusions.

**Results file:** `results/Mahak/ablation_counting_methods_v2.csv`

## Experiment 3  Sensitivity to the Number of Kernels and Groups

### Objective

Investigate how different combinations of the Hydra parameters `k` and `g` affect classification accuracy, macro F1-score, and runtime.

### Experimental Setup

Three configurations will be compared while keeping `k × g = 512` constant:

| Configuration |  k |   g |
| ------------- | -: | --: |
| k4_g128       |  4 | 128 |
| k8_g64        |  8 |  64 |
| k16_g32       | 16 |  32 |

The experiment will use the Coffee, ItalyPowerDemand, and Wafer datasets, with multiple random seeds where computationally practical.

### Evaluation Metrics

* Classification accuracy
* Macro F1-score
* Runtime in seconds


In [14]:

# Experiment 3: Sensitivity to k and g

KG_RESULTS_FILE = RESULTS_DIR / "ablation_kg_sensitivity.csv"

KG_CONFIGS = [
    {"name": "k4_g128", "k": 4, "g": 128},
    {"name": "k8_g64", "k": 8, "g": 64},
    {"name": "k16_g32", "k": 16, "g": 32},
]

KG_DATASETS = ["Coffee", "ItalyPowerDemand", "Wafer"]

# Use 3 seeds for an initial comparison.
KG_SEEDS = [0, 1, 2]

kg_results = []

for dataset_name in KG_DATASETS:
    for config in KG_CONFIGS:
        for seed in KG_SEEDS:
            print(
                f"Running {dataset_name} | {config['name']} | seed={seed}"
            )

            result = evaluate_hydra_variant(
                dataset_name=dataset_name,
                variant_name=config["name"],
                hydra_class=Hydra,
                seed=seed,
                k=config["k"],
                g=config["g"],
                batch_size=BATCH_SIZE,
            )

            kg_results.append(result)

kg_df = pd.DataFrame(kg_results)

print(f"\nTotal results collected: {len(kg_df)}")
display(kg_df)

kg_df.to_csv(KG_RESULTS_FILE, index=False)
print(f"\nResults saved to: {KG_RESULTS_FILE}")



Running Coffee | k4_g128 | seed=0
Running Coffee | k4_g128 | seed=1
Running Coffee | k4_g128 | seed=2
Running Coffee | k8_g64 | seed=0
Running Coffee | k8_g64 | seed=1
Running Coffee | k8_g64 | seed=2
Running Coffee | k16_g32 | seed=0
Running Coffee | k16_g32 | seed=1
Running Coffee | k16_g32 | seed=2
Running ItalyPowerDemand | k4_g128 | seed=0
Running ItalyPowerDemand | k4_g128 | seed=1
Running ItalyPowerDemand | k4_g128 | seed=2
Running ItalyPowerDemand | k8_g64 | seed=0
Running ItalyPowerDemand | k8_g64 | seed=1
Running ItalyPowerDemand | k8_g64 | seed=2
Running ItalyPowerDemand | k16_g32 | seed=0
Running ItalyPowerDemand | k16_g32 | seed=1
Running ItalyPowerDemand | k16_g32 | seed=2
Running Wafer | k4_g128 | seed=0
Running Wafer | k4_g128 | seed=1
Running Wafer | k4_g128 | seed=2
Running Wafer | k8_g64 | seed=0
Running Wafer | k8_g64 | seed=1
Running Wafer | k8_g64 | seed=2
Running Wafer | k16_g32 | seed=0
Running Wafer | k16_g32 | seed=1
Running Wafer | k16_g32 | seed=2

Total res

,dataset,variant,seed,k,g,accuracy,macro_f1,runtime_seconds
0,Coffee,k4_g128,0,4,128,1.000000,1.000000,0.289118
1,Coffee,k4_g128,1,4,128,1.000000,1.000000,0.277969
2,Coffee,k4_g128,2,4,128,1.000000,1.000000,0.294517
3,Coffee,k8_g64,0,8,64,1.000000,1.000000,0.310499
4,Coffee,k8_g64,1,8,64,1.000000,1.000000,0.269321
5,Coffee,k8_g64,2,8,64,1.000000,1.000000,0.258763
6,Coffee,k16_g32,0,16,32,1.000000,1.000000,0.239044
7,Coffee,k16_g32,1,16,32,1.000000,1.000000,0.237639
8,Coffee,k16_g32,2,16,32,1.000000,1.000000,0.247021
9,ItalyPowerDemand,k4_g128,0,4,128,0.968902,0.968902,0.316714



Results saved to: /Users/shreed/PROJECTS/Hydra_1/results/Mahak/ablation_kg_sensitivity.csv


### Results

Three configurations (`k4_g128`, `k8_g64`, `k16_g32`, with k × g fixed at 512) were evaluated on Coffee, ItalyPowerDemand and Wafer using seeds 0, 1 and 2. Values are means over the 3 seeds.

| Dataset | Configuration | Mean accuracy | Std | Mean macro F1 | Mean runtime (s) |
| ------- | ------------- | ------------: | --: | ------------: | ---------------: |
| Coffee | k4_g128 | 1.0000 | 0.0000 | 1.0000 | 0.287 |
| Coffee | k8_g64 | 1.0000 | 0.0000 | 1.0000 | 0.280 |
| Coffee | k16_g32 | 1.0000 | 0.0000 | 1.0000 | **0.241** |
| ItalyPowerDemand | k4_g128 | 0.9666 | 0.0031 | 0.9666 | 0.301 |
| ItalyPowerDemand | k8_g64 | **0.9689** | 0.0029 | **0.9689** | 0.287 |
| ItalyPowerDemand | k16_g32 | 0.9653 | 0.0024 | 0.9653 | **0.282** |
| Wafer | k4_g128 | 1.0000 | 0.0000 | 1.0000 | 12.888 |
| Wafer | k8_g64 | 1.0000 | 0.0000 | 1.0000 | 12.011 |
| Wafer | k16_g32 | 1.0000 | 0.0000 | 1.0000 | **11.544** |

### Conclusion

Coffee and Wafer reached 100% accuracy with every configuration and seed. On ItalyPowerDemand, the default `k8_g64` had the highest mean accuracy (96.89%). The spread across configurations (0.36 percentage points) is about the same size as the seed-to-seed standard deviation (about 0.3 points), so the difference is not meaningful.

Runtime fell slightly as k increased: `k16_g32` was the fastest configuration on all three datasets (about 4% faster than `k8_g64` on Wafer).

Overall, the k/g split has little effect on accuracy for these datasets, and the default `k8_g64` is a reasonable choice. These findings are limited to three datasets that are near 100% accuracy, and three seeds. The results agree with the 10-seed k/g experiment in `shreed_hydra_baseline.ipynb`.

**Results file:** `results/Mahak/ablation_kg_sensitivity.csv`

## FINAL SUMMARY

# Mahak - Hydra Ablation Experiments

## 1. Objective

To investigate how changes to Hydra's counting mechanism (max-only, hard vs soft counting) and its k/g configuration affect time series classification accuracy and runtime.

## 2. Experimental Setup

* **Model:** the authors' original Hydra implementation (`hydra-authors-repo/code/hydra.py`). Variants are subclasses; the original file is unchanged.
* **Pipeline:** Hydra transform, then `SparseScaler`, then `RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))`. This is the same classifier setting as the authors and the other team notebooks.
* **Data:** UCR datasets with the predefined train/test splits.
* **Metrics:** accuracy, macro F1-score and runtime (seconds, covering transform, scaling, fitting and prediction).

**Re-run note:** this notebook was re-run after fixing two issues.
* The `HydraCountingVariant` methods were not indented inside the class.
* The classifier used the default alphas instead of `np.logspace(-3, 3, 10)`.

The results below come from the corrected run.

## 3. Experiment 1 - Max-Only vs. Max-and-Min

On Coffee (seed 0), both the original and the max-only variant reached 100% accuracy and a macro F1 of 1.00. Max-only uses half the features (3072 vs 6144) and was faster (0.225 s vs 0.282 s).

The result is preliminary because there is one dataset and one seed, and the dataset is already at 100% accuracy.

## 4. Experiment 2 - Counting Method Comparison

| Dataset | hard_only | soft_only | combined_default (original Hydra) |
| ------- | --------: | --------: | --------------------------------: |
| Coffee | 1.00 | 1.00 | 1.00 |
| ECG200 | 0.84 | 0.87 | 0.87 |
| GunPoint | 1.00 | 1.00 | 1.00 |

* Hard counting on both channels was the worst variant (−3 percentage points on ECG200).
* Soft counting on both channels tied exactly with the original Hydra.
* The earlier finding that `soft_only` beat the original (90% vs 89%) did not hold up after the fixes.

## 5. Experiment 3 - Parameter Sensitivity

| Configuration | k | g | k × g | Coffee | ItalyPowerDemand | Wafer |
| ------------- | -: | --: | ----: | -----: | ---------------: | ----: |
| k4_g128 | 4 | 128 | 512 | 1.0000 | 0.9666 | 1.0000 |
| k8_g64 | 8 | 64 | 512 | 1.0000 | **0.9689** | 1.0000 |
| k16_g32 | 16 | 32 | 512 | 1.0000 | 0.9653 | 1.0000 |

Values are mean accuracy over seeds 0–2.

The differences between configurations are within the seed-to-seed variation. `k16_g32` was the fastest configuration on all three datasets.

## 6. Output Files

Results are stored under `results/Mahak/`:

* `ablation_max_min.csv`
* `ablation_counting_methods_v2.csv`
* `ablation_kg_sensitivity.csv`

## 7. Overall Conclusion

None of the tested modifications improved accuracy over the original Hydra on these datasets.

* **Original design choices hold up.** Soft counting for the max channel matters: replacing it with hard counting cost 3 percentage points on ECG200. Switching the min channel from hard to soft counting made no difference.
* **The default k/g setting is fine.** Changing the k/g split (with k × g = 512) had no meaningful effect on accuracy. Larger k (`k16_g32`) is slightly faster.
* **Max-only is a possible efficiency gain.** It halves the feature count and reduced runtime with no loss of accuracy on Coffee. It is worth testing on harder datasets.

**Limitations:**
* Most experiments used three datasets, most of which are already at or near 100% accuracy.
* Experiments 1 and 2 used a single seed.
* With so little room to improve, these experiments cannot detect small effects.

**Next steps:**
* Repeat the ablations on harder UCR datasets (for example Phoneme, InlineSkate, Haptics, EthanolLevel and the EOG datasets) or on all 112 datasets, with multiple seeds or resamples.
* Run the planned first-order differencing and clipping ablations, which have not been done yet.